# Credit Risk Analytics & Loan Approval Prediction — Runnable Notebook

**What this notebook contains:**

- Synthetic dataset generation (10,000 rows)
- SQL-style ETL demonstration using SQLite and CTEs
- Preprocessing with scikit-learn pipelines
- Model training (Logistic Regression, Random Forest)
- Evaluation (ROC, PR, confusion matrix)
- Saving trained pipelines and artifacts

This notebook is runnable end-to-end on a local machine or cloud instance with Python. It was generated on: 2025-11-18 22:18:00 UTC

## 0. Setup & Requirements

Run this cell to install required packages if you don't have them. (If you use a conda environment, install equivalent packages.)

```bash
pip install numpy pandas scikit-learn matplotlib joblib nbformat
```

In [ ]:
# Imports
import numpy as np
import pandas as pd
import sqlite3, os, joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, precision_recall_curve, auc
import matplotlib.pyplot as plt
%matplotlib inline
print('Libraries loaded')

## 1. Generate synthetic loan dataset

This creates a plausible synthetic loans dataset (10,000 rows) with features commonly used in credit risk modelling.

In [ ]:
# Create synthetic dataset
np.random.seed(42)
n = 10000
emp_options = list(range(0,31))
probs = np.array([0.07 if i==0 else 0.03 if i<=10 else 0.01 for i in emp_options], dtype=float)
probs = probs / probs.sum()

df = pd.DataFrame({
    "loan_id": np.arange(1, n+1),
    "loan_amount": np.random.randint(2000, 50000, size=n),
    "term_months": np.random.choice([36, 60], size=n, p=[0.7, 0.3]),
    "interest_rate": np.round(np.random.normal(12, 4, size=n).clip(4, 30), 2),
    "annual_income": np.round(np.random.normal(60000, 25000, size=n).clip(8000, 300000), 2),
    "emp_length_years": np.random.choice(emp_options, size=n, p=probs),
    "credit_score": np.round(np.random.normal(680, 60, size=n).clip(300, 850)).astype(int),
    "dti": np.round(np.random.normal(18, 8, size=n).clip(0, 60),2),
    "num_open_accounts": np.random.poisson(5, size=n).clip(0, 40),
    "delinquencies_2yrs": np.random.poisson(0.3, size=n).clip(0, 10),
    "purpose": np.random.choice(["debt_consolidation","credit_card","home_improvement","major_purchase","small_business","other"], size=n, p=[0.45,0.25,0.1,0.08,0.06,0.06]),
    "home_ownership": np.random.choice(["RENT","OWN","MORTGAGE","OTHER"], size=n, p=[0.45,0.3,0.22,0.03]),
    "age": np.random.randint(18, 75, size=n)
})

# Missingness
for col in ["annual_income", "credit_score", "dti"]:
    mask = np.random.rand(n) < 0.02
    df.loc[mask, col] = np.nan

# Simulate default probability and target
logit = (
    -4.5
    + 0.00002 * df["loan_amount"] 
    + 0.05 * (df["interest_rate"] - 10)
    - 0.006 * (df["credit_score"] - 650)
    + 0.04 * (df["dti"] - 15).fillna(0)
    + 0.5 * df["delinquencies_2yrs"] 
    + 0.01 * (df["num_open_accounts"] - 5)
)
prob_default = 1 / (1 + np.exp(-logit))
df['prob_default_sim'] = prob_default
df['default'] = (np.random.rand(n) < prob_default).astype(int)

os.makedirs('data', exist_ok=True)
df.to_csv('data/loan_data_sample.csv', index=False)
df.head()

## 2. SQL-style ETL using SQLite (CTEs)

We demonstrate an in-database transformation using a CTE to produce `credit_band` and `dti_band`. This mirrors how you'd build feature tables in a production database.

In [ ]:
# Use SQLite in-memory DB to demonstrate SQL pipeline
conn = sqlite3.connect(':memory:')
df.to_sql('loans_raw', conn, index=False, if_exists='replace')

sql = """
WITH cleaned AS (
  SELECT
    loan_id, loan_amount, term_months, interest_rate,
    COALESCE(annual_income, -1) AS annual_income,
    COALESCE(credit_score, -1) AS credit_score,
    COALESCE(dti, -1) AS dti,
    emp_length_years, num_open_accounts, delinquencies_2yrs, purpose, home_ownership, age,
    "default" AS is_default, prob_default_sim
  FROM loans_raw
),
features AS (
  SELECT
    *,
    CASE WHEN credit_score >= 750 THEN 'excellent'
         WHEN credit_score >= 700 THEN 'good'
         WHEN credit_score >= 650 THEN 'fair'
         WHEN credit_score >= 600 THEN 'poor'
         ELSE 'very_poor' END AS credit_band,
    CASE WHEN dti < 10 THEN 'low'
         WHEN dti < 20 THEN 'medium'
         WHEN dti < 35 THEN 'high'
         ELSE 'very_high' END AS dti_band
  FROM cleaned
)
SELECT loan_id, loan_amount, interest_rate, annual_income, credit_score, dti, credit_band, dti_band, is_default, prob_default_sim
FROM features LIMIT 10;
"""

sample_sql = pd.read_sql_query(sql, conn)
sample_sql.to_csv('data/sql_pipeline_sample.csv', index=False)
sample_sql

## 3. Preprocessing and model training

We build scikit-learn pipelines with numeric imputation + scaling and categorical one-hot encoding. Then train Logistic Regression and Random Forest.

In [ ]:
# Create bands in pandas (mirror SQL)
def band_credit(x):
    if pd.isna(x): return 'very_poor'
    x = int(x)
    if x >= 750: return 'excellent'
    if x >= 700: return 'good'
    if x >= 650: return 'fair'
    if x >= 600: return 'poor'
    return 'very_poor'

def band_dti(x):
    if pd.isna(x): return 'very_high'
    x = float(x)
    if x < 10: return 'low'
    if x < 20: return 'medium'
    if x < 35: return 'high'
    return 'very_high'

df['credit_band'] = df['credit_score'].apply(band_credit)
df['dti_band'] = df['dti'].apply(band_dti)

num_cols = ['loan_amount','term_months','interest_rate','annual_income','emp_length_years','credit_score','dti','num_open_accounts','delinquencies_2yrs','age']
cat_cols = ['purpose','home_ownership','credit_band','dti_band']

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['default'], random_state=42)

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value='missing')),('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, num_cols), ('cat', categorical_transformer, cat_cols)])

X_train = train_df[num_cols + cat_cols]
y_train = train_df['default']
X_test = test_df[num_cols + cat_cols]
y_test = test_df['default']

pipe_lr = Pipeline(steps=[('preprocessor', preprocessor), ('clf', LogisticRegression(max_iter=1000))])
pipe_rf = Pipeline(steps=[('preprocessor', preprocessor), ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])

pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

# Save pipelines
os.makedirs('models', exist_ok=True)
joblib.dump(pipe_lr, 'models/logistic_pipeline.joblib')
joblib.dump(pipe_rf, 'models/randomforest_pipeline.joblib')

print('Models trained and saved to models/')

## 4. Evaluation: ROC, PR curve, Confusion matrix, Feature importance

In [ ]:
# Predictions and metrics
y_proba_lr = pipe_lr.predict_proba(X_test)[:,1]
y_proba_rf = pipe_rf.predict_proba(X_test)[:,1]

auc_lr = roc_auc_score(y_test, y_proba_lr)
auc_rf = roc_auc_score(y_test, y_proba_rf)

print('ROC AUC - Logistic:', round(auc_lr,3), 'RandomForest:', round(auc_rf,3))

# ROC plot
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
plt.figure(figsize=(8,6))
plt.plot(fpr_lr, tpr_lr, label=f'LR (AUC={auc_lr:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'RF (AUC={auc_rf:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set'); plt.legend(); plt.grid(True)
plt.savefig('data/roc_curve.png'); plt.show()

# Confusion matrix for RF at threshold 0.5
y_pred_rf = (y_proba_rf >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_rf)
print('Confusion Matrix (RF, thresh=0.5):\n', cm)

plt.figure(figsize=(4,3))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.xticks([0,1], ['No Default','Default']); plt.yticks([0,1], ['No Default','Default'])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha='center', va='center', color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.title('Confusion Matrix'); plt.colorbar(); plt.savefig('data/confusion_matrix.png'); plt.show()

# Precision-Recall curve for RF
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_proba_rf)
pr_auc_rf = auc(recall_rf, precision_rf)
plt.figure(figsize=(6,5))
plt.plot(recall_rf, precision_rf, label=f'RF (PR AUC={pr_auc_rf:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve'); plt.legend(); plt.grid(True)
plt.savefig('data/pr_curve.png'); plt.show()

# Feature importance (RF)
cat_onehot = pipe_rf.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_features = list(cat_onehot.get_feature_names_out(cat_cols))
feature_names = num_cols + cat_features
importances = pipe_rf.named_steps['clf'].feature_importances_
feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
feat_imp.to_csv('data/feature_importance.csv', index=False)
feat_imp.head(15)

## 5. Save summary and artifacts

This saves model summary CSV and shows where to find saved files.

In [ ]:
summary = {'model': ['LogisticRegression', 'RandomForest'], 'roc_auc_test': [auc_lr, auc_rf], 'pr_auc_test': [pr_auc_rf, pr_auc_rf]}
summary_df = pd.DataFrame(summary)
summary_df.to_csv('data/model_summary.csv', index=False)
print('Saved artifacts in data/ and models/ directories:')
for root, dirs, files in os.walk('.', topdown=True):
    if root.startswith('./data') or root.startswith('./models'):
        for f in files:
            print(os.path.join(root, f))

## 6. How to use this notebook

1. Open the notebook in JupyterLab or Jupyter Notebook.
2. Run cells sequentially. The `data/` folder will contain generated CSVs and plots. The `models/` folder contains saved pipelines.
3. To use the saved model for scoring, load with `joblib.load('models/randomforest_pipeline.joblib')` and call `.predict_proba(df_to_score)`.

---

If you want, replace the synthetic data in `data/loan_data_sample.csv` with your real dataset (same column names) and re-run cells from the SQL or preprocessing step onward.